<a href="https://colab.research.google.com/github/iav2002/Assignment_Advanced_Topics_In_DeepLearning/blob/main/Part2_Experiments_3(DifferentialLearningRates).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 3 — Differential Learning Rates on ResNet50

Tests whether using different learning rates for the pretrained backbone and the new classification head improves over the uniform-LR full fine-tuning of Variant B in exp1.

 The intuition: the head is initialized randomly and benefits from a higher LR, while the backbone holds delicate pretrained features that need gentler updates. A uniform LR forces a compromise between these two needs.



## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type == 'cuda':
    print(f'gpu: {torch.cuda.get_device_name(0)}')

Mounted at /content/drive
device: cuda
gpu: NVIDIA A100-SXM4-40GB


## 2. Paths and shared training config

Same training config as Exp 1 and Exp 2 for direct comparison. The 3 LR ratios live in `LR_CONFIGS` so the sweep loop reads from one place.

In [2]:
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdvancedDL')
INDEX_DIR = DRIVE_ROOT / 'sample_index'
RESULTS_DIR = DRIVE_ROOT / 'results' / 'exp3_differential_lr'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 0 # avoiding
NUM_EPOCHS = 20
WEIGHT_DECAY = 1e-4
PATIENCE = 5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# the 3 LR configurations - head fixed at 1e-3, backbone varies
# label encodes the ratio for easy reading later
LR_CONFIGS = [
    {'label': 'ratio_10x',  'lr_head': 1e-3, 'lr_backbone': 1e-4},
    {'label': 'ratio_20x',  'lr_head': 1e-3, 'lr_backbone': 5e-5},
    {'label': 'ratio_100x', 'lr_head': 1e-3, 'lr_backbone': 1e-5},
]

print(f'results dir: {RESULTS_DIR}')
print(f'configs to test:')
for c in LR_CONFIGS:
    ratio = c['lr_head'] / c['lr_backbone']
    print(f"  {c['label']:12s}: head={c['lr_head']:.0e}, backbone={c['lr_backbone']:.0e} (ratio {ratio:.0f}x)")

results dir: /content/drive/MyDrive/Colab Notebooks/AdvancedDL/results/exp3_differential_lr
configs to test:
  ratio_10x   : head=1e-03, backbone=1e-04 (ratio 10x)
  ratio_20x   : head=1e-03, backbone=5e-05 (ratio 20x)
  ratio_100x  : head=1e-03, backbone=1e-05 (ratio 100x)


## 3. Copy dataset to local SSD + load sample index

Same pipeline as Exp 1 and Exp 2. Skips fast if cache exists.

In [3]:
import shutil

LOCAL_ROOT = Path('/content/dataset_local')
DRIVE_RAW = DRIVE_ROOT / 'raw'
EXPECTED = {'train': 4654, 'val': 1125, 'test': 1124}

def copy_split(split):
    src = DRIVE_RAW / split / 'images'
    dst = LOCAL_ROOT / split / 'images'
    dst.mkdir(parents=True, exist_ok=True)
    if len(list(dst.glob('*.png'))) == EXPECTED[split]:
        print(f'  {split}: already complete')
        return
    print(f'  {split}: copying {EXPECTED[split]} images...')
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'  {split}: copied in {time.time()-t0:.0f}s')

print('copying dataset to local SSD...')
for split in ['train', 'val', 'test']:
    copy_split(split)
print('done.')

copying dataset to local SSD...
  train: copying 4654 images...
  train: copied in 256s
  val: copying 1125 images...
  val: copied in 99s
  test: copying 1124 images...
  test: copied in 102s
done.


In [4]:
with open(INDEX_DIR / 'samples_train.json') as f: samples_train = json.load(f)
with open(INDEX_DIR / 'samples_val.json') as f:   samples_val = json.load(f)
with open(INDEX_DIR / 'samples_test.json') as f:  samples_test = json.load(f)
with open(INDEX_DIR / 'class_vocab.json') as f:   vocab = json.load(f)

CLASS_TO_IDX = vocab['class_to_idx']
IDX_TO_CLASS = {int(k): v for k, v in vocab['idx_to_class'].items()}
NUM_CLASSES = len(CLASS_TO_IDX)
CLASSES = vocab['keep_classes']

for split_samples in [samples_train, samples_val, samples_test]:
    for s in split_samples:
        s['img_path'] = s['img_path'].replace(str(DRIVE_RAW), str(LOCAL_ROOT))

print(f'classes: {NUM_CLASSES}')
print(f'train: {len(samples_train):,}  val: {len(samples_val):,}  test: {len(samples_test):,}')

classes: 11
train: 11,000  val: 2,750  test: 2,750


## 4. Pre-load to RAM (Copy from previous notebooks)

In [5]:
def preload_to_ram(samples):
    n = len(samples)
    imgs = torch.empty((n, 3, IMG_SIZE, IMG_SIZE), dtype=torch.uint8)
    labels = torch.empty((n,), dtype=torch.long)
    t0 = time.time()
    for i, s in enumerate(samples):
        img = Image.open(s['img_path']).convert('RGB')
        x1, y1, x2, y2 = s['bbox']
        crop = img.crop((x1, y1, x2, y2)).resize((IMG_SIZE, IMG_SIZE))
        imgs[i] = torch.from_numpy(np.asarray(crop)).permute(2, 0, 1)
        labels[i] = s['label']
        if (i+1) % 2000 == 0:
            print(f'  {i+1:,}/{n:,}')
    print(f'done: {n:,} in {time.time()-t0:.0f}s')
    return imgs, labels

class InMemoryDataset(Dataset):
    def __init__(self, imgs, labels):
        self.imgs = imgs; self.labels = labels
        self.mean = torch.tensor(IMAGENET_MEAN).view(3,1,1) * 255
        self.std = torch.tensor(IMAGENET_STD).view(3,1,1) * 255
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        x = self.imgs[i].float()
        return (x - self.mean) / self.std, self.labels[i]

CACHE_DIR = Path('/content/dataset_local')
def preload_or_load(samples, name):
    path = CACHE_DIR / f'preloaded_{name}.pt'
    if path.exists():
        print(f'{name}: loading cache...')
        cache = torch.load(path)
        return cache['imgs'], cache['labels']
    print(f'{name}: preloading...')
    imgs, labels = preload_to_ram(samples)
    torch.save({'imgs': imgs, 'labels': labels}, path)
    return imgs, labels

train_imgs, train_labels = preload_or_load(samples_train, 'train')
val_imgs, val_labels = preload_or_load(samples_val, 'val')
test_imgs, test_labels = preload_or_load(samples_test, 'test')

train_loader = DataLoader(InMemoryDataset(train_imgs, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(InMemoryDataset(val_imgs, val_labels),     batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(InMemoryDataset(test_imgs, test_labels),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'loaders ready')

train: preloading...


/tmp/ipykernel_943/987304223.py:10: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  imgs[i] = torch.from_numpy(np.asarray(crop)).permute(2, 0, 1)


  2,000/11,000
  4,000/11,000
  6,000/11,000
  8,000/11,000
  10,000/11,000
done: 11,000 in 147s
val: preloading...
  2,000/2,750
done: 2,750 in 37s
test: preloading...
  2,000/2,750
done: 2,750 in 37s
loaders ready


## 5. Shared helpers

Same train/eval loops as previous experiments. The new piece is `make_diff_lr_optimizer`, which builds AdamW with two parameter groups, one for the head (at the high LR) and one for the backbone (at the low LR). PyTorch handles the rest: each param group keeps its own LR through the scheduler.

In [ ]:
import torchvision.models as models
from torch.amp import autocast, GradScaler

def build_model(num_classes):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

# the key new helper: separate the head's params from the backbone's
# named_parameters lets us route by name - anything starting with "fc" goes to head group
def make_diff_lr_optimizer(model, lr_head, lr_backbone, weight_decay):
    head_params = [p for n, p in model.named_parameters() if n.startswith('fc') and p.requires_grad]
    backbone_params = [p for n, p in model.named_parameters() if not n.startswith('fc') and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': head_params, 'lr': lr_head, 'name': 'head'},
        {'params': backbone_params, 'lr': lr_backbone, 'name': 'backbone'},
    ], weight_decay=weight_decay)
    return optimizer

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', dtype=torch.float16):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n += xb.size(0)
    return total_loss / total_n, total_correct / total_n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    per_class_correct = torch.zeros(NUM_CLASSES)
    per_class_total = torch.zeros(NUM_CLASSES)
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
        with autocast(device_type='cuda', dtype=torch.float16):
            logits = model(xb)
            loss = criterion(logits, yb)
        preds = logits.argmax(1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_n += xb.size(0)
        for c in range(NUM_CLASSES):
            mask = (yb == c)
            per_class_total[c] += mask.sum().item()
            per_class_correct[c] += (preds[mask] == c).sum().item()
    per_class_acc = (per_class_correct / per_class_total.clamp(min=1)).tolist()
    return total_loss / total_n, total_correct / total_n, per_class_acc

print('helpers defined')